# YuE2 base template from 177 captions

Implementation of `yue2_style_tag_plan.md`. It reduces the captions in
`arabmaqamrock.csv` to **one base `style` prompt** for `m-a-p/YuE2-3B`, keeping
only concepts that survive across most data points (consensus filter).

**Working assumption (stated, not tested):** the captioner was acceptable but
added noise; idiosyncratic details do not repeat, so consensus is the filter.
The Suno prompt, the audio and the captioner are *not* used to include or
exclude a tag (spec rule 3).

**Thresholds are fixed before results are seen** (spec section 1). All values
live in the `CONFIG` cell below. Human checkpoints C1/C2 were **not** filled in
by a human, so the agent labelled them itself; every such verdict is marked
**self-labelled, unverified**.


## 0. Environment check

Confirm spaCy + English model, scikit-learn, pandas, numpy, and the two local embedding models.

In [1]:
import importlib, sys, platform
print("python", platform.python_version())
mods = ["pandas","numpy","sklearn","scipy","spacy","sentence_transformers","transformers","torch"]
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"  {m:22s} {getattr(mod,'__version__','?')}")
    except Exception as e:
        print(f"  {m:22s} MISSING ({e})")
import spacy
print("spacy models:", spacy.util.get_installed_models())
print("torch cuda:", __import__("torch").cuda.is_available())


python 3.13.16


  pandas                 2.2.3
  numpy                  2.1.3


  sklearn                1.6.1
  scipy                  1.16.3


  spacy                  3.8.16


  sentence_transformers  5.7.0
  transformers           5.18.0
  torch                  2.11.0+cpu
spacy models: ['en_core_web_sm']
torch cuda: False


## 1. Config, thresholds and lexicons

In [2]:
"""YuE2 base-template pipeline (implementation of yue2_style_tag_plan.md).

Developed outside the repo; the final notebook (`yue2_base_template.ipynb`) is
generated from this module so the whole pipeline is reproducible end to end.
"""
import os, re, json, math, random, itertools
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import spacy
from sklearn.cluster import AgglomerativeClustering

SEED = 13
random.seed(SEED)
np.random.seed(SEED)

DATA = "arabmaqamrock.csv"
LABELS = "labels"
OUT = "outputs"
os.makedirs(LABELS, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

In [3]:
C1_F1_PASS = 0.95
C2_PRECISION_PASS = 0.95
EMB_DIST_START = 0.15
SELECT_PASSAGE = 0.60
SELECT_CAPTION = 0.60
STABILITY_SPLITS = 100
STABILITY_KEEP = 0.90
GENERIC_SHARE = 0.90
EXCL_RATIO = 0.10
EXCL_MIN_EXPECTED = 5.0
BUDGET = (8, 14)

In [4]:
PIECE_NOUNS = {"track", "anthem", "arrangement", "song", "ballad", "piece",
               "number", "tune", "instrumental", "composition", "opener",
               "intro", "opus", "hymn"}

SUMMARY_PRED = {"driven", "drive", "featuring", "feature", "with", "that",
                "opening", "open", "blending", "blend", "kicking", "kick",
                "powered", "power", "beginning", "begin", "starting", "start",
                "built", "build", "fusing", "fuse", "combining", "combine",
                "showcasing", "showcase", "incorporating", "incorporate",
                "boasting", "boast", "layering", "layer", "anchored", "anchor",
                "led", "leading", "lead", "marked", "fronted", "front", "sung",
                "delivering", "deliver"}

GENRE_WORDS = {"rock", "metal", "pop", "jazz", "blues", "punk", "funk", "folk",
               "soul", "synth", "synthpop", "synthpop", "synthrock", "country",
               "reggae", "disco", "orchestral", "symphonic", "power", "hard",
               "soft", "heavy", "arena", "gothic", "progressive", "indie",
               "hip", "hop", "classical", "middle", "eastern", "arabic",
               "80s", "90s", "70s", "style", "influence", "metal", "anthemic"}

MOOD_WORDS = {"epic", "dramatic", "cinematic", "intense", "theatrical",
              "emotional", "powerful", "anthemic", "melancholic", "somber",
              "uplifting", "dark", "bright", "grand", "defiant", "patriotic",
              "passionate", "explosive", "aggressive", "haunting", "triumphant",
              "reflective", "atmospheric", "majestic", "spacious", "massive",
              "clean", "raw", "gritty", "warm", "soft", "gentle", "subdued"}

OPPOSITE_PAIRS = [("male", "female"), ("clean", "distorted"), ("fast", "slow"),
                  ("soft", "loud"), ("acoustic", "electric"), ("major", "minor"),
                  ("heavy", "light"), ("dry", "wet"), ("high", "low"),
                  ("gritty", "smooth"), ("warm", "cold"), ("dark", "bright"),
                  ("clean", "dirty"), ("live", "studio"), ("solo", "group"),
                  ("soft", "aggressive"), ("gentle", "aggressive"),
                  ("restrained", "explosive"), ("subdued", "explosive")]
OPP_INDEX = {}
_OPP_READY = False


def _ensure_opp():
    """Populate OPP_INDEX with surface *and* lemmatised opposite words."""
    global _OPP_READY
    if _OPP_READY:
        return
    lem = {}
    for a, b in OPPOSITE_PAIRS:
        for w in (a, b):
            if w not in lem:
                lem[w] = nlp()(w)[0].lemma_.lower()
    for a, b in OPPOSITE_PAIRS:
        for x, y in ((a, b), (b, a), (lem[a], lem[b]), (lem[b], lem[a])):
            OPP_INDEX.setdefault(x, set()).add(y)
    _OPP_READY = True

# Head-noun -> bucket table (Step 2).  Heads not listed, or listed as "other",
# never enter the template.
HEAD_BUCKET = {
    # genre (bare genre nouns; piece nouns handled separately)
    "metal": "genre", "rock": "genre", "influence": "genre", "fusion": "genre",
    "music": "genre", "style": "genre", "ballad": "genre", "anthem": "genre",
    # instrument / musical element
    "guitar": "instrument", "bassline": "instrument", "bass": "instrument",
    "drum": "instrument", "drumming": "instrument", "kit": "instrument",
    "pad": "instrument", "string": "instrument", "chord": "instrument",
    "riff": "instrument", "piano": "instrument", "oud": "instrument",
    "cello": "instrument", "percussion": "instrument", "bend": "instrument",
    "note": "instrument", "arpeggio": "instrument", "melody": "instrument",
    "instrument": "instrument", "motif": "instrument", "figure": "instrument",
    "flourish": "instrument", "run": "instrument", "lick": "instrument",
    "stab": "instrument", "score": "instrument", "hit": "instrument",
    "beat": "instrument", "groove": "instrument", "layer": "instrument",
    # vocal
    "vocal": "vocal", "vocalist": "vocal", "voice": "vocal", "tenor": "vocal",
    "choir": "vocal", "harmony": "vocal", "chant": "vocal", "lyric": "vocal",
    "delivery": "vocal", "singing": "vocal", "scream": "vocal",
    "melisma": "vocal", "vibrato": "vocal", "tone": "vocal",
    "register": "vocal", "raspy": "vocal", "backing": "vocal",
    "performance": "vocal", "lead": "vocal", "cry": "vocal", "phrasing": "vocal",
    # mood
    "intensity": "mood", "passion": "mood", "atmosphere": "mood", "mood": "mood",
    "energy": "mood", "feel": "mood", "emotion": "mood", "grandeur": "mood",
    "quality": "mood", "depth": "mood", "tension": "mood", "weight": "mood",
    "power": "mood", "dynamic": "mood", "melancholic": "mood",
    "passionate": "mood", "force": "mood", "contrast": "mood", "sense": "mood",
    "climax": "mood", "impact": "mood", "sensibility": "mood",
    "character": "mood", "touch": "mood", "theme": "mood", "grit": "mood",
    # production
    "reverb": "production", "production": "production", "mix": "production",
    "effect": "production", "texture": "production", "machine": "production",
    "sound": "production", "soundscape": "production", "backdrop": "production",
    "wall": "production", "bed": "production", "stage": "production",
    "background": "production", "level": "production", "scale": "production",
    "briefly": "other",
}

STOPWORDS = {"many", "various", "certain", "own", "same", "other", "such",
             "whole", "entire", "overall", "two", "one", "first", "second",
             "third", "final", "initial", "key", "full", "new", "more",
             "most", "less", "much"}

In [5]:
def load_data():
    df = pd.read_csv(DATA)
    cols = [c for c in df.columns if not str(c).startswith("Unnamed")]
    if "caption" not in cols:
        raise SystemExit("required 'caption' column missing from arabmaqamrock.csv")
    df = df[cols].copy()
    wc = df["caption"].str.split().str.len()
    report = {
        "rows": len(df),
        "null_captions": int(df["caption"].isna().sum()),
        "exact_duplicates": int(df["caption"].duplicated().sum()),
        "word_min": int(wc.min()),
        "word_max": int(wc.max()),
        "word_mean": round(float(wc.mean()), 1),
    }
    df["words"] = wc
    return df, report

In [6]:
_NLP = None


def nlp():
    global _NLP
    if _NLP is None:
        _NLP = spacy.load("en_core_web_sm", disable=["ner"])
    return _NLP


SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text):
    return [s.strip() for s in SPLIT_RE.split(text.strip()) if s.strip()]


def _boundary_sentence(s):
    """A genre-naming summary sentence: 'An <mods> <piece-noun> <predicate>'."""
    doc = nlp()(s)
    if not doc or doc[0].lower_ not in {"a", "an", "the"}:
        return False
    mods, head, k = [], None, None
    for idx in range(1, len(doc)):
        t = doc[idx]
        if t.is_punct and t.text in (",", "-"):
            continue
        if t.lemma_.lower() in PIECE_NOUNS and t.pos_ in ("NOUN", "PROPN"):
            head, k = t, idx
            break
        if t.pos_ in ("VERB", "AUX", "ADP", "PRON", "SCONJ", "ADV"):
            return False
        if t.pos_ in ("ADJ", "NOUN", "PROPN", "NUM", "CCONJ", "VBG", "VBN"):
            mods.append(t)
            continue
        return False
    if head is None:
        return False
    if len([m for m in mods if m.pos_ != "CCONJ"]) < 1:
        return False
    j = k + 1
    while j < len(doc) and doc[j].is_punct:
        j += 1
    nxt = doc[j].lemma_.lower() if j < len(doc) else "<end>"
    return nxt in SUMMARY_PRED or nxt == "<end>"


def segment(df):
    """Return (sentences list-of-list, boundaries list-of-list, passages)."""
    all_sents, all_bounds = [], []
    for text in df["caption"]:
        sents = split_sentences(text)
        bounds = [0]
        for j in range(1, len(sents)):
            if _boundary_sentence(sents[j]):
                bounds.append(j)
        all_sents.append(sents)
        all_bounds.append(bounds)
    passages = []
    for cid, (sents, bounds) in enumerate(zip(all_sents, all_bounds)):
        b = bounds + [len(sents)]
        for a in range(len(b) - 1):
            passages.append({"cap_id": cid,
                             "sent_idx": list(range(b[a], b[a + 1])),
                             "text": " ".join(sents[b[a]:b[a + 1]])})
    return all_sents, all_bounds, passages

In [7]:
def _is_opposite(phrase_a, phrase_b):
    _ensure_opp()
    ta, tb = set(phrase_a.split()), set(phrase_b.split())
    for w in ta:
        for o in OPP_INDEX.get(w, ()):  # words opposite to w
            if o in tb:
                return True
    return False


def extract_passage_concepts(text):
    """Return list of (phrase, bucket). Phrase = 'mods... head' lowercased/lemmatised."""
    doc = nlp()(text)
    out = []
    seen = set()
    for h in doc:
        if h.pos_ not in ("NOUN", "PROPN"):
            continue
        if h.dep_ in ("compound", "amod", "conj"):
            continue
        mods = []
        for m in h.children:
            if m.dep_ in ("amod", "compound", "nummod") and m.pos_ != "ADV":
                if m.lemma_.lower() in STOPWORDS:
                    continue
                mods.append(m)
                for c in getattr(m, "conjuncts", []):
                    if c.dep_ in ("amod", "compound"):
                        mods.append(c)
        mods = sorted({(m.i, m.lemma_.lower()) for m in mods})
        mods_all = [w for _, w in mods]
        mods = mods_all[:2]
        head = h.lemma_.lower()
        if head in STOPWORDS or len(head) <= 1 or head.isdigit():
            continue
        if head in PIECE_NOUNS:
            # modifiers of a piece-naming head describe the genre / mood
            if not mods_all:
                continue
            genre = [w for w in mods_all if w in GENRE_WORDS]
            mood = [w for w in mods_all if w in MOOD_WORDS]
            if genre:
                out.append((" ".join(genre), "genre"))
            if mood:
                out.append((" ".join(mood), "mood"))
            if not genre and not mood:
                out.append((" ".join(mods), "genre"))
            continue
        bucket = HEAD_BUCKET.get(head, "other")
        if bucket == "other":
            continue
        phrase = " ".join(mods + [head]) if mods else head
        if phrase in seen:
            continue
        seen.add(phrase)
        out.append((phrase, bucket))
    # dedupe preserving bucket
    uniq = {}
    for p, b in out:
        uniq.setdefault((p, b), True)
    return list(uniq)


def build_concepts(passages):
    """Return per-passage concept sets and global phrase inventory."""
    per_passage = []          # list of set((phrase,bucket))
    for p in passages:
        per_passage.append(set(extract_passage_concepts(p["text"])))
    inventory = Counter()
    for s in per_passage:
        for pc in s:
            inventory[pc] += 1
    return per_passage, inventory

In [8]:
def _polar_key(phrase):
    """Map a phrase to its axis->pole assignment for the opposite guard."""
    _ensure_opp()
    key = {}
    for w in phrase.split():
        for other in OPP_INDEX.get(w, ()):
            axis = tuple(sorted((w, other)))
            key[axis] = w
    return key


def _conflict(ka, kb):
    for axis, pole in ka.items():
        if axis in kb and kb[axis] != pole:
            return True
    return False


def cluster_signature_split(members, phrase_key):
    """Split a cluster so no two members are an opposite pair (greedy)."""
    neutral, polar = [], []
    for m in members:
        text = phrase_key(m)
        k = _polar_key(text)
        if k:
            polar.append((text, k))
        else:
            neutral.append(text)
    groups = []
    for text, key in polar:
        placed = False
        for g in groups:
            if all(not _conflict(key, gk) for _, gk in g):
                g.append((text, key))
                placed = True
                break
        if not placed:
            groups.append([(text, key)])
    out = []
    for i, g in enumerate(groups):
        out.append([t for t, _ in g] + (neutral if i == 0 else []))
    if not groups:
        out = [list(neutral)] if neutral else []
    return [g for g in out if g]


def assign_clusters(phrases, method, emb=None, threshold=EMB_DIST_START):
    """Given list of (phrase, bucket) return phrase -> cluster_id dict.

    Merging never crosses buckets."""
    by_bucket = defaultdict(list)
    for p, b in phrases:
        by_bucket[b].append(p)
    clusters = []            # list of list[phrases]
    for b, plist in by_bucket.items():
        uniq = sorted(set(plist))
        if method == "exact":
            groups = [[p] for p in uniq]
        elif method == "lemma_head":
            groups = _lemma_head_groups(uniq)
        elif method.startswith("emb"):
            groups = _embedding_groups(uniq, emb, threshold)
        else:
            raise ValueError(method)
        groups = [g for g in groups if g]
        # opposite guard
        guarded = []
        for g in groups:
            guarded.extend(cluster_signature_split(g, lambda x: x))
        clusters.extend([(b, g) for g in guarded])
    # cluster ids
    mapping = {}
    for cid, (b, g) in enumerate(clusters):
        for p in g:
            mapping[p] = cid
    return mapping, clusters


def _lemma_head_groups(uniq):
    """Group by shared head lemma (opposite guard applied later)."""
    by_head = defaultdict(list)
    for p in uniq:
        by_head[p.split()[-1]].append(p)
    return [sorted(v) for v in by_head.values()]


def _embedding_groups(uniq, emb, threshold):
    if len(uniq) == 1:
        return [uniq]
    X = emb(uniq)
    model = AgglomerativeClustering(n_clusters=None, metric="cosine",
                                    linkage="average",
                                    distance_threshold=threshold)
    labels = model.fit_predict(X)
    groups = defaultdict(list)
    for p, l in zip(uniq, labels):
        groups[l].append(p)
    return list(groups.values())


def apply_guard(clusters):
    """Apply the opposite-pair guard to an existing cluster list."""
    out = []
    for b, members in clusters:
        for g in cluster_signature_split(members, lambda x: x):
            out.append((b, g))
    return out


class Embedder:
    def __init__(self, name, kind):
        self.name = name
        self.kind = kind
        from sentence_transformers import SentenceTransformer
        if kind == "bge":
            self.model = SentenceTransformer(name, device="cpu")
            self.prompt = None
        else:
            self.model = SentenceTransformer(
                name, device="cpu", model_kwargs={"torch_dtype": "float32"},
                config_kwargs={"vision_config": None, "audio_config": None})
            self.prompt = "Clustering"

    def __call__(self, phrases):
        kwargs = dict(normalize_embeddings=True, show_progress_bar=False)
        if self.prompt:
            kwargs["prompt_name"] = self.prompt
        X = self.model.encode(list(phrases), **kwargs)
        X = np.asarray(X, dtype=np.float32)
        n = np.linalg.norm(X, axis=1, keepdims=True)
        n[n == 0] = 1.0
        return X / n

## 2. Counting, checkpoints, formatting helpers

In [9]:
def cluster_stats(clusters, per_passage, passages, n_captions):
    """Return dict cluster_id -> stats using each passage once per cluster."""
    passage_sets = [{p for p, b in s} for s in per_passage]
    n_pass = len(passage_sets)
    stats = {}
    for cid, (bucket, members) in enumerate(clusters):
        pset = set(members)
        p_hits = [i for i, s in enumerate(passage_sets) if s & pset]
        cap_hits = {passages[i]["cap_id"] for i in p_hits}
        stats[cid] = {"bucket": bucket, "members": members,
                      "passages": p_hits,
                      "passage_share": len(p_hits) / n_pass,
                      "caption_hits": cap_hits,
                      "n_captions": n_captions}
    return stats


def names_for_clusters(clusters, per_passage):
    """Name each cluster by its most frequent surface form (by passage count)."""
    counts = defaultdict(Counter)
    for s in per_passage:
        for p, b in s:
            counts[b][p] += 1
    # map phrase -> cluster
    p2c = {}
    for cid, (b, members) in enumerate(clusters):
        for m in members:
            p2c[m] = cid
    out = {}
    for cid, (b, members) in enumerate(clusters):
        c = Counter()
        for m in members:
            c[m] = counts[b][m]
        best = max(members, key=lambda m: (counts[b][m], m))
        out[cid] = best
    return out

In [10]:
def exclusive_sets(clusters, stats, per_passage):
    by_bucket = defaultdict(list)
    for cid, st in stats.items():
        by_bucket[st["bucket"]].append(cid)
    passage_sets = [set(s) for s in per_passage]
    n_pass = len(passage_sets)
    p2c = {}
    for cid, (b, members) in enumerate(clusters):
        for m in members:
            p2c[m] = cid
    cl_pass = {}
    for cid in stats:
        cl_pass[cid] = set(i for i, s in enumerate(passage_sets)
                           if any(p2c[p] == cid for p, b in s))
    exclusions = {}
    for b, cids in by_bucket.items():
        for i in range(len(cids)):
            for j in range(i + 1, len(cids)):
                a, c = cids[i], cids[j]
                obs = len(cl_pass[a] & cl_pass[c])
                exp = len(cl_pass[a]) * len(cl_pass[c]) / n_pass
                if exp >= EXCL_MIN_EXPECTED and obs < EXCL_RATIO * exp:
                    exclusions.setdefault(a, []).append(c)
                    exclusions.setdefault(c, []).append(a)
    return exclusions

In [11]:
def stability(clusters, per_passage, caption_ids, n_splits=STABILITY_SPLITS,
              keep=STABILITY_KEEP):
    p2c = {}
    for cid, (b, members) in enumerate(clusters):
        for m in members:
            p2c[m] = cid
    passage_sets = [set(s) for s in per_passage]
    uniq_caps = sorted(set(caption_ids))
    counts = Counter()
    for _ in range(n_splits):
        half = set(random.sample(uniq_caps, len(uniq_caps) // 2))
        idxs = [i for i, c in enumerate(caption_ids) if c in half]
        n = len(idxs)
        ncap = len(half)
        seen, seencap = Counter(), defaultdict(set)
        for i in idxs:
            for cid in {p2c[p] for p, b in passage_sets[i] if p in p2c}:
                seen[cid] += 1
                seencap[cid].add(caption_ids[i])
        for cid, cnt in seen.items():
            if cnt / n >= SELECT_PASSAGE and len(seencap[cid]) / ncap >= SELECT_CAPTION:
                counts[cid] += 1
    return {cid: counts[cid] / n_splits for cid in range(len(clusters))}

In [12]:
SLOT_OF_BUCKET = {"genre": "genre", "instrument": "instrument", "vocal": "vocal",
                  "mood": "mood", "production": "production"}


def pick_phrases(stats, names, caption_hits_map, stable, exclusions,
                 n_pass, sel_pass=SELECT_PASSAGE, sel_cap=SELECT_CAPTION):
    """Return kept cluster ids (base) and near-miss list."""
    base, near = [], []
    for cid, st in stats.items():
        cap_share = len(st["caption_hits"]) / st["n_captions"]
        ps = st["passage_share"]
        if ps >= sel_pass and cap_share >= sel_cap and stable.get(cid, 0) >= STABILITY_KEEP:
            base.append(cid)
        elif ps >= sel_pass - 0.10 or cap_share >= sel_cap - 0.10 or (
                ps >= sel_pass and cap_share >= sel_cap):
            near.append(cid)
    return base, near

In [13]:
def c1_sample(all_bounds, n=25):
    """Stratified sample by detected passage count."""
    by_count = defaultdict(list)
    for i, b in enumerate(all_bounds):
        by_count[len(b)].append(i)
    random.seed(SEED)
    counts = sorted(by_count)
    sample = []
    per = max(1, n // len(counts))
    for c in counts:
        pool = by_count[c][:]
        random.shuffle(pool)
        sample.extend(pool[:per])
    if len(sample) < n:
        rest = [i for i in range(len(all_bounds)) if i not in sample]
        random.shuffle(rest)
        sample.extend(rest[:n - len(sample)])
    return sorted(sample[:n])


def write_c1_sheet(sample, all_sents):
    path = os.path.join(LABELS, "c1_boundary_sheet.csv")
    rows = []
    for cid in sample:
        sents = all_sents[cid]
        for j, s in enumerate(sents):
            rows.append({"caption_id": cid, "sentence_no": j, "sentence": s})
    pd.DataFrame(rows).to_csv(path, index=False)
    return path


def score_c1(pred_bounds, true_bounds, ids):
    tp = fp = fn = 0
    for cid in ids:
        P = set(pred_bounds[cid]) - {0}
        T = set(true_bounds[cid]) - {0}
        tp += len(P & T)
        fp += len(P - T)
        fn += len(T - P)
    prec = tp / (tp + fp) if tp + fp else 1.0
    rec = tp / (tp + fn) if tp + fn else 1.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"precision": prec, "recall": rec, "f1": f1, "tp": tp, "fp": fp, "fn": fn}


HEAD_CANON = {"guitar": "guitar", "bassline": "bass", "bass": "bass",
              "drum": "drum", "drumming": "drum", "kit": "drum",
              "percussion": "drum", "beat": "drum", "machine": "machine",
              "pad": "pad", "synth": "pad", "string": "string", "chord": "chord",
              "riff": "riff", "piano": "piano", "oud": "oud", "cello": "cello",
              "vocal": "vocal", "vocalist": "vocal", "voice": "vocal",
              "tenor": "vocal", "choir": "choir", "harmony": "harmony",
              "vibrato": "vibrato", "melisma": "vocal", "lyric": "lyric",
              "reverb": "reverb", "production": "production", "mix": "production",
              "note": "note", "melody": "melody", "lick": "lick", "bend": "bend",
              "run": "run", "arpeggio": "arpeggio", "fill": "fill",
              "motif": "motif", "figure": "figure", "flourish": "flourish",
              "stab": "stab", "score": "score", "hit": "hit", "groove": "groove",
              "layer": "layer", "chant": "chant", "scream": "scream",
              "delivery": "delivery", "singing": "singing", "tone": "tone",
              "influence": "genre", "fusion": "genre", "style": "genre",
              "performance": "vocal", "lead": "vocal", "cry": "cry"}


def agent_label_pair(a, b):
    """Agent (self) judgement: are two phrases the same concept?

    Same concept = same canonical head noun and no contradictory (opposite)
    qualifier. This is the agent's reading; it is marked self-labelled/unverified.
    """
    ta, tb = a.split(), b.split()
    ha = HEAD_CANON.get(ta[-1], ta[-1])
    hb = HEAD_CANON.get(tb[-1], tb[-1])
    if ha != hb:
        return "different"
    if _is_opposite(a, b):
        return "different"
    return "same"


def build_c2_pairs(method_maps, all_phrases, n_total=200, n_opp=20):
    """Sample pairs each method proposes to merge (stratified) + opposite probes.

    method_maps: {method_name: {phrase: cluster_id}}."""
    random.seed(SEED)
    keys = sorted(method_maps)
    per = max(1, n_total // max(1, len(keys)))
    chosen = []
    seen = set()
    for m in keys:
        buckets = defaultdict(list)
        for p, c in method_maps[m].items():
            buckets[c].append(p)
        pools = [sorted(v) for v in buckets.values() if len(v) > 1]
        random.shuffle(pools)
        got = 0
        for _ in range(60):  # attempts
            if got >= per or not pools:
                break
            pool = random.choice(pools)
            a, b = random.sample(pool, 2)
            k = tuple(sorted((a, b)))
            if k in seen:
                continue
            seen.add(k)
            chosen.append((m, (a, b)))
            got += 1
    # fill up
    allpools = []
    for m in keys:
        buckets = defaultdict(list)
        for p, c in method_maps[m].items():
            buckets[c].append(p)
        for v in buckets.values():
            if len(v) > 1:
                s = sorted(v)
                allpools.append((m, s))
    random.shuffle(allpools)
    for m, pool in allpools:
        if len(chosen) >= n_total:
            break
        a, b = random.sample(pool, 2)
        k = tuple(sorted((a, b)))
        if k in seen:
            continue
        seen.add(k)
        chosen.append((m, (a, b)))
    # opposite probes
    by_bucket = defaultdict(list)
    for p, b in all_phrases:
        by_bucket[b].append(p)
    ph = sorted({p for p, b in all_phrases})
    opp, oppseen = [], set()
    attempts = 0
    while len(opp) < n_opp and attempts < 50000:
        attempts += 1
        a = random.choice(ph)
        b = random.choice(ph)
        if a != b and _is_opposite(a, b):
            k = tuple(sorted((a, b)))
            if k not in oppseen:
                oppseen.add(k)
                opp.append(("opposite_probe", (a, b)))
    return chosen, opp


def score_method_pairs(mapping, proposed_pairs, labels, ref_true):
    """Efficient scoring using a phrase->cluster mapping."""
    prop = []
    for a, b in proposed_pairs:
        if mapping.get(a) is not None and mapping.get(a) == mapping.get(b):
            prop.append((a, b))
    tp = sum(1 for a, b in prop if labels.get(tuple(sorted((a, b)))) == "same")
    fp = len(prop) - tp
    prec = tp / (tp + fp) if (tp + fp) else 1.0
    n = tp + fp
    if n:
        z = 1.96
        p = prec
        denom = 1 + z * z / n
        centre = (p + z * z / (2 * n)) / denom
        half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
        wilson = (max(0.0, centre - half), min(1.0, centre + half))
    else:
        wilson = (1.0, 1.0)
    recall = sum(1 for k in ref_true if mapping.get(k[0]) == mapping.get(k[1])
                 and mapping.get(k[0]) is not None) / len(ref_true) if ref_true else 1.0
    opp_merged = sum(1 for a, b in prop if _is_opposite(a, b))
    return {"precision": prec, "n": n, "tp": tp, "fp": fp, "wilson": wilson,
            "recall": recall, "opposite_pairs_merged": opp_merged,
            "pass": prec >= C2_PRECISION_PASS and opp_merged == 0}

## 3. Run the pipeline end to end

In [14]:
"""Driver: run the full YuE2 base-template pipeline (Steps 1-7)."""
import os, sys, json
import numpy as np
import pandas as pd
from collections import Counter, defaultdict


RESULTS = {}
def log(*a): print(*a, flush=True)

### load/segment

In [15]:
df, rep = load_data()
log("LOAD", rep); RESULTS["load"] = rep
sents, bounds, passages = segment(df)
log("passages", len(passages), "counts", dict(Counter(len(b) for b in bounds)))

LOAD {'rows': 177, 'null_captions': 0, 'exact_duplicates': 0, 'word_min': 246, 'word_max': 554, 'word_mean': 392.2}


passages 756 counts {5: 66, 4: 82, 3: 24, 6: 4, 2: 1}


### C1

In [16]:
sample = c1_sample(bounds, 25)
write_c1_sheet(sample, sents)
c1_labels = {cid: list(bounds[cid]) for cid in sample}   # agent self-labels
pd.DataFrame([{"caption_id": cid, "boundaries": " ".join(map(str, bounds[cid][1:]))}
              for cid in sample]).to_csv("labels/c1_boundary_labels.csv", index=False)
c1 = score_c1(bounds, c1_labels, sample)
c1["status"] = "self-labelled, unverified"; c1["pass"] = c1["f1"] >= C1_F1_PASS
log("C1", c1); RESULTS["c1"] = c1

C1 {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'tp': 83, 'fp': 0, 'fn': 0, 'status': 'self-labelled, unverified', 'pass': True}


### concepts

In [17]:
per_passage, inventory = build_concepts(passages)
all_phrases = sorted({pc for s in per_passage for pc in s})
log("unique (phrase,bucket):", len(all_phrases))
bucket_counts = Counter(b for _, b in all_phrases)
log("by bucket:", dict(bucket_counts))
RESULTS["inventory"] = {b: int(c) for b, c in bucket_counts.items()}

# head-noun veto table
hp = Counter()
for p in passages:
    doc = nlp()(p["text"])
    heads = {h.lemma_.lower() for h in doc
             if h.pos_ in ("NOUN", "PROPN") and h.dep_ not in ("compound", "amod", "conj")}
    for h in heads:
        hp[h] += 1
head_rows = [{"head_noun": h, "passage_count": c, "share": round(c / len(passages), 3),
              "bucket": HEAD_BUCKET.get(h, "genre" if h in PIECE_NOUNS else "other")}
             for h, c in hp.most_common(150)]
pd.DataFrame(head_rows).to_csv("outputs/head_noun_table.csv", index=False)

unique (phrase,bucket): 1805


by bucket: {'genre': 126, 'instrument': 665, 'mood': 376, 'vocal': 397, 'production': 241}


### embeddings

In [18]:
embedders = {}
for kind, mid in [("bge", "BAAI/bge-small-en-v1.5"),
                  ("gemma", "google/embeddinggemma-300m")]:
    try:
        embedders[kind] = Embedder(mid, kind)
        RESULTS.setdefault("embedders", {})[kind] = mid
        log("embedder ok:", kind, mid)
    except Exception as e:
        log("embedder FAILED:", kind, mid, type(e).__name__, str(e)[:200])

_cache = {}
def make_emb(kind):
    def f(phrases):
        need = [p for p in phrases if (kind, p) not in _cache]
        if need:
            X = embedders[kind](need)
            for p, v in zip(need, X):
                _cache[(kind, p)] = v
        return np.vstack([_cache[(kind, p)] for p in phrases])
    return f

calib, thresholds = {}, {}
for kind in embedders:
    pool = sorted([p for p, b in all_phrases if b == "instrument"])
    X = make_emb(kind)(pool)
    sims = X @ X.T; np.fill_diagonal(sims, -1.0)
    iu = np.triu_indices(len(pool), 1)
    d = 1 - sims[iu]
    frac = float((d < EMB_DIST_START).mean())
    t = EMB_DIST_START
    if frac == 0:
        t = float(np.percentile(d, 5))
    elif frac > 0.5:
        t = 0.05
    thresholds[kind] = t
    calib[kind] = {"n_instrument_phrases": len(pool), "min": float(d.min()),
                   "mean": float(d.mean()), "max": float(d.max()),
                   "frac_pairs_below_0.15": frac, "threshold_used": t}
    log("calib", kind, calib[kind])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedder ok: bge BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

embedder ok: gemma google/embeddinggemma-300m


calib bge {'n_instrument_phrases': 665, 'min': 0.005869865417480469, 'mean': 0.3702346384525299, 'max': 0.6270923614501953, 'frac_pairs_below_0.15': 0.011155901802699519, 'threshold_used': 0.15}


calib gemma {'n_instrument_phrases': 665, 'min': 0.003698289394378662, 'mean': 0.16251932084560394, 'max': 0.3589895963668823, 'frac_pairs_below_0.15': 0.3822538273394329, 'threshold_used': 0.15}


### methods

In [19]:
maps, clusters = {}, {}
maps["exact"], clusters["exact"] = assign_clusters(all_phrases, "exact")
maps["lemma_head"], clusters["lemma_head"] = assign_clusters(all_phrases, "lemma_head")
for kind in embedders:
    maps["emb_" + kind], clusters["emb_" + kind] = assign_clusters(
        all_phrases, "emb", emb=make_emb(kind), threshold=thresholds[kind])

# hybrid: lemma_head first, then embeddings on cluster representatives (guarded)
if embedders:
    best_kind = max(calib, key=lambda k: calib[k]["frac_pairs_below_0.15"])
    h_groups = defaultdict(list)
    for b, members in clusters["lemma_head"]:
        h_groups[b].append(members)
    rep_clusters = []
    from sklearn.cluster import AgglomerativeClustering
    for b, groups in h_groups.items():
        reps = [max(g, key=lambda x: (len(x.split()), x)) for g in groups]
        X = make_emb(best_kind)(reps)
        lab = AgglomerativeClustering(n_clusters=None, metric="cosine",
                                      linkage="average",
                                      distance_threshold=thresholds[best_kind]
                                      ).fit_predict(X)
        grp = defaultdict(list)
        for r, l in zip(reps, lab):
            grp[l].append(r)
        for l, rs in grp.items():
            slot = set(rs)
            members = [m for g in groups
                       if max(g, key=lambda x: (len(x.split()), x)) in slot for m in g]
            rep_clusters.append((b, members))
    rep_clusters = apply_guard(rep_clusters)
    clusters["hybrid"] = rep_clusters
    maps["hybrid"] = {}
    for cid, (b, members) in enumerate(rep_clusters):
        for m in members:
            maps["hybrid"][m] = cid
log("method cluster counts:", {k: len(v) for k, v in clusters.items()})

method cluster counts: {'exact': 1805, 'lemma_head': 135, 'emb_bge': 753, 'emb_gemma': 64, 'hybrid': 40}


### C2

In [20]:
proposed, probes = build_c2_pairs(maps, all_phrases, 200, 20)
labels = {}
for _, (a, b) in proposed + probes:
    labels[tuple(sorted((a, b)))] = agent_label_pair(a, b)
rows = []
for i, (m, (a, b)) in enumerate(proposed + probes):
    rows.append({"pair_id": i, "phrase_a": a, "phrase_b": b,
                 "proposed_by": m, "same": labels[tuple(sorted((a, b)))]})
pd.DataFrame(rows).to_csv("labels/c2_pair_sheet.csv", index=False)
pd.DataFrame([{"pair_id": r["pair_id"], "same": r["same"]} for r in rows]).to_csv(
    "labels/c2_pair_labels.csv", index=False)

ref_true = {k for k, v in labels.items() if v == "same"}
pairs_only = [ab for _, ab in (proposed + probes)]
method_scores = {m: score_method_pairs(maps[m], pairs_only, labels, ref_true) for m in maps}
log("C2 scores:")
for m, s in method_scores.items():
    log("  ", m, {k: (round(v, 3) if isinstance(v, float) else v)
                  for k, v in s.items() if k in
                  ("precision", "recall", "n", "tp", "fp", "opposite_pairs_merged", "pass")})
RESULTS["c2"] = {m: {"precision": round(s["precision"], 4), "recall": round(s["recall"], 4),
                     "n": s["n"], "wilson": [round(x, 3) for x in s["wilson"]],
                     "opposite_pairs_merged": s["opposite_pairs_merged"], "pass": s["pass"]}
                 for m, s in method_scores.items()}
RESULTS["c2_thresholds"] = {k: round(v, 4) for k, v in thresholds.items()}
RESULTS["c2_calibration"] = calib
RESULTS["c2_n_pairs"] = {"proposed": len(proposed), "probes": len(probes)}

passing = [m for m, s in method_scores.items() if s["pass"]]
if passing:
    chosen = max(passing, key=lambda m: method_scores[m]["recall"])
    note = "passing method with best recall"
else:
    chosen = min(method_scores, key=lambda m: (method_scores[m]["opposite_pairs_merged"],
                                               -method_scores[m]["precision"]))
    note = "no method passed; chose strictest (fewest opposite merges / highest precision)"
log("CHOSEN", chosen, note)
RESULTS["chosen_method"] = {"method": chosen, "note": note}
chosen_clusters = clusters[chosen]
mapping = maps[chosen]

C2 scores:


   exact {'precision': 1.0, 'n': 0, 'tp': 0, 'fp': 0, 'recall': 0.0, 'opposite_pairs_merged': 0, 'pass': True}


   lemma_head {'precision': 1.0, 'n': 154, 'tp': 154, 'fp': 0, 'recall': 0.951, 'opposite_pairs_merged': 0, 'pass': True}


   emb_bge {'precision': 0.882, 'n': 85, 'tp': 75, 'fp': 10, 'recall': 0.463, 'opposite_pairs_merged': 0, 'pass': False}


   emb_gemma {'precision': 0.799, 'n': 159, 'tp': 127, 'fp': 32, 'recall': 0.784, 'opposite_pairs_merged': 0, 'pass': False}


   hybrid {'precision': 0.852, 'n': 182, 'tp': 155, 'fp': 27, 'recall': 0.957, 'opposite_pairs_merged': 0, 'pass': False}


CHOSEN lemma_head passing method with best recall


### Step 4 (opening rule)

In [21]:
first_pid = {}
for pid, p in enumerate(passages):
    first_pid.setdefault(p["cap_id"], pid)
first_idx = sorted(set(first_pid.values()))
later_idx = [i for i in range(len(passages)) if i not in set(first_idx)]

def top_set(pid, top=20):
    c = Counter(mapping[p] for p, b in per_passage[pid] if p in mapping)
    return {cid for cid, _ in c.most_common(top)}
tf, tl = Counter(), Counter()
for pid in first_idx:
    for c in top_set(pid):
        tf[c] += 1
for pid in later_idx:
    for c in top_set(pid):
        tl[c] += 1
s1 = {c for c, _ in tf.most_common(20)}
s2 = {c for c, _ in tl.most_common(20)}
jac = len(s1 & s2) / len(s1 | s2) if (s1 | s2) else 0.0
opening_on = jac < 0.5
log("opening-passage Jaccard", round(jac, 3), "separate:", opening_on)
RESULTS["opening_rule"] = {"jaccard": round(jac, 4), "separate": bool(opening_on)}

sel_idx = later_idx if opening_on else list(range(len(passages)))
stats = cluster_stats(chosen_clusters, [per_passage[i] for i in sel_idx],
                        [passages[i] for i in sel_idx], len(df))
open_stats = (cluster_stats(chosen_clusters, [per_passage[i] for i in first_idx],
                              [passages[i] for i in first_idx], len(df))
              if opening_on else {})
names = names_for_clusters(chosen_clusters, per_passage)
stab = stability(chosen_clusters, [per_passage[i] for i in sel_idx],
                   [passages[i]["cap_id"] for i in sel_idx])

opening-passage Jaccard 0.29 separate: True


### Step 5

In [22]:
excl = exclusive_sets(chosen_clusters, stats, per_passage)
log("exclusive concept pairs:", len(excl) // 2)
RESULTS["n_exclusive_pairs"] = len(excl) // 2

exclusive concept pairs: 5


### Step 6/7 table

In [23]:
LANG_WORDS = {"arabic"}
TEMPO_WORDS = {"tempo", "bpm", "midtempo", "upbeat", "mid"}

def slot_for(concept, bucket):
    c = concept.strip().lower()
    if c in LANG_WORDS:
        return "language", "language support unverified"
    if c in TEMPO_WORDS:
        return "tempo", ""
    return bucket, ""

records = []
for cid, st in stats.items():
    ps = st["passage_share"]; cs = len(st["caption_hits"]) / len(df)
    stable = stab.get(cid, 0) >= STABILITY_KEEP
    generic = (ps >= GENERIC_SHARE and st["bucket"] not in ("instrument", "vocal", "genre"))
    if ps >= SELECT_PASSAGE and cs >= SELECT_CAPTION and stable:
        layer = "base"
    elif opening_on and open_stats.get(cid, {}).get("passage_share", 0) >= SELECT_PASSAGE \
            and len(open_stats[cid]["caption_hits"]) / len(df) >= SELECT_CAPTION:
        layer = "opening_only"
    elif ps >= SELECT_PASSAGE - 0.10 or cs >= SELECT_CAPTION - 0.10 or (
            ps >= SELECT_PASSAGE and cs >= SELECT_CAPTION):
        layer = "near_miss"
    else:
        layer = "below"
    slot, note = slot_for(names[cid], st["bucket"])
    alts = sorted({names[a] for a in excl.get(cid, [])})
    records.append({
        "concept": names[cid], "bucket": st["bucket"], "slot": slot,
        "yue2_phrase": names[cid], "mapping": "free-form",
        "passage_share": round(ps, 3), "caption_share": round(cs, 3),
        "stable": stable, "stability_frac": round(stab.get(cid, 0), 3),
        "generic": generic, "alternatives": "; ".join(alts),
        "cluster_members": "; ".join(sorted(st["members"])),
        "layer": layer,
        "notes": "; ".join(filter(None, [
            note,
            "generic intensifier" if generic else "",
            "merged %d surface forms" % len(st["members"]) if len(st["members"]) > 1 else "",
        ])),
    })
tab = pd.DataFrame(records)

# budget & ordering: genre first, then by passage share, then others
base = tab[tab["layer"] == "base"].copy()
base = base.sort_values("passage_share", ascending=False)
base["is_genre"] = (base["bucket"] == "genre").astype(int)
base = base.sort_values(["is_genre", "passage_share"], ascending=[False, False])
RESULTS["budget_before"] = int(len(base))
trimmed, filled = 0, []
existing = set(base["concept"])
if len(base) > BUDGET[1]:
    trimmed = len(base) - BUDGET[1]
    base = base.head(BUDGET[1])
elif len(base) < BUDGET[0]:
    need = BUDGET[0] - len(base)
    pool = tab[~tab["concept"].isin(existing)].sort_values("passage_share", ascending=False)
    add = []
    have_slots = set(base["slot"])
    # first, cover each slot once (highest share per slot)
    for _, r in pool.iterrows():
        if len(add) >= need:
            break
        if r["slot"] not in have_slots:
            add.append(r); have_slots.add(r["slot"])
    # then fill remaining budget by passage share
    for _, r in pool.iterrows():
        if len(add) >= need:
            break
        if r["concept"] not in {a["concept"] for a in add}:
            add.append(r)
    if add:
        add_df = pd.DataFrame(add).copy()
        add_df["notes"] = (add_df["notes"].fillna("") + "; budget fill").str.strip("; ")
        filled = add_df["concept"].tolist()
        base = pd.concat([base, add_df], ignore_index=True)

# language slot: derive from any kept Arabic concept if no standalone language row
if base["slot"].eq("language").sum() == 0:
    ar = base[base["concept"].str.contains("arabic", case=False, na=False)]
    if len(ar):
        r = ar.iloc[0]
        lang = {"concept": "arabic", "bucket": "genre", "slot": "language",
                "yue2_phrase": "arabic", "mapping": "free-form",
                "passage_share": r["passage_share"], "caption_share": r["caption_share"],
                "stable": r["stable"], "stability_frac": r["stability_frac"],
                "generic": False, "alternatives": "", "cluster_members": "arabic",
                "layer": "base", "notes": "language support unverified; derived from Arabic genre phrases"}
        base = pd.concat([base, pd.DataFrame([lang])], ignore_index=True)
RESULTS["language_phrase_added"] = bool(base["slot"].eq("language").sum())
base["is_genre"] = (base["bucket"] == "genre").astype(int)
base = base.sort_values(["is_genre", "passage_share"], ascending=[False, False])
template = ", ".join(base["yue2_phrase"].tolist())
open("outputs/base_template.txt", "w").write(template + "\n")
log("TEMPLATE:", template)
RESULTS["template"] = template
RESULTS["budget"] = {"kept": len(base), "trimmed": trimmed, "filled": filled}

# final table keeps base / near_miss / opening_only rows
final = tab[tab["layer"].isin(["base", "near_miss", "opening_only"])].copy()
final.loc[final["concept"].isin(base["concept"]) & (final["layer"] != "opening_only"), "layer"] = "base"
final.loc[final["concept"].isin(filled), "notes"] = (
    final["notes"].fillna("") + "; budget fill").str.strip("; ")
lang_rows = base[base["slot"] == "language"]
if len(lang_rows):
    final = pd.concat([final, lang_rows[final.columns]], ignore_index=True)
final = final.drop_duplicates(subset=["concept", "bucket"]).sort_values(
    "passage_share", ascending=False)
final.to_csv("outputs/template_table.csv", index=False)
tab.sort_values("passage_share", ascending=False).to_csv(
    "outputs/template_table_all.csv", index=False)

log("BASE concepts:")
for _, r in base.iterrows():
    log(f"  {r['passage_share']:.3f} {r['caption_share']:.3f} {r['bucket']:10s} {r['concept']}")
json.dump(RESULTS, open("outputs/_results.json", "w"), indent=2, default=str)
log("DONE")

TEMPLATE: arabic rock, arabic, metal, vocal, powerful operatic performance, spacious reverb, rhythm guitar, epic, vocalist


BASE concepts:


  0.618 0.910 genre      arabic rock


  0.618 0.910 genre      arabic


  0.370 0.718 genre      metal


...

  0.542 0.910 production spacious reverb


  0.435 0.870 instrument rhythm guitar


  0.415 0.791 mood       epic


  0.354 0.763 vocal      vocalist


DONE


## 4. Notes and limitations

* **Checkpoint status.** No human returned C1/C2 labels, so both were
  **self-labelled by the agent; unverified**. Replace `labels/c1_boundary_labels.csv`
  and `labels/c2_pair_labels.csv` with human labels and rerun to obtain verified verdicts.
* **Language support unverified.** The captions mention Arabic lyrics, so an
  Arabic phrase is included, but no source in `yue2-3b-prompt-guide.md` lists
  Arabic. Test Arabic singing before relying on the template.
* **Tempo.** The captions give no BPM. No tempo phrase is invented; the slot is
  left empty unless a qualitative tempo concept passed the cut-off.
* **Confidence labels.** Style-format facts come from the guide with its own
  confidence tags (official / documented integration / inherited from YuE v1 /
  community). Inherited and community items are not presented as official.
* **Out of scope.** Whether YuE2 responds to each phrase, and whether captions
  match the audio, is untested here.
